# The Information Systems Atlas — OpenAlex collector

This notebook builds a resumable, graph-ready dataset for MIS Quarterly, Information Systems Research, Journal of Management Information Systems, and Journal of the Association for Information Systems. It reads `OpenAlex_API_KEY` from the root `.env` file and writes every output beside this notebook.

**Outputs created while it runs**

- `mis_quarterly_papers.json`, `information_systems_research_papers.json`, `journal_of_management_information_systems_papers.json`, and `journal_of_the_association_for_information_systems_papers.json`: full OpenAlex records for focal-journal papers plus incoming/outgoing citation IDs.
- `atlas_external_papers.json`: minimal records (title, journal, year, DOI/link) for citing/referenced papers outside the four focal journals.
- `atlas_graph_edges.json`: deduplicated directed citation edges (`source_id` cites `target_id`).
- `atlas_state.json`: resumable checkpoint. Do not delete this unless deliberately starting over.

Run the setup cell, then the collection cell. Re-running the collection cell continues unfinished journals and skips completed work. Interrupting the kernel is safe: completed API pages and batches are already on disk. Every three months, set `START_QUARTERLY_REFRESH = True` once in the collection cell, run it, then turn it back to `False`: this re-checks works, incoming/outgoing citations, and metadata for changes. The four primary records retain `openalex_raw` as well as a dashboard-friendly normalized view; external records intentionally remain minimal.

OpenAlex returns abstracts (where available) as an inverted index rather than plain text. The collector reconstructs that index and stores plain text only for the four focal journals.

In [1]:
# Setup and reusable helpers
from __future__ import annotations

import json
import os
import time
from uuid import uuid4
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

try:
    import requests
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError("Install the notebook dependencies once with: %pip install requests tqdm") from exc

ROOT = Path.cwd().resolve()
ENV_PATH = ROOT / '.env'
if not ENV_PATH.exists():
    raise FileNotFoundError(f'Expected {ENV_PATH}. Start Jupyter from the project root.')

# A tiny local .env loader avoids requiring python-dotenv and never prints the API key.
for raw_line in ENV_PATH.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if not line or line.startswith('#') or '=' not in line:
        continue
    key, value = line.split('=', 1)
    os.environ.setdefault(key.strip(), value.strip().strip(chr(34)).strip(chr(39)))

API_KEY = os.getenv('OpenAlex_API_KEY')
if not API_KEY:
    raise RuntimeError('OpenAlex_API_KEY was not found in .env.')

API_URL = 'https://api.openalex.org'
REQUEST_TIMEOUT_SECONDS = 90
PER_PAGE = 200
REQUEST_PAUSE_SECONDS = 0.10  # gentle pacing; change only if your API plan permits it
MAX_RETRIES = 7
MAX_WRITE_RETRIES = 12
REFERENCE_CHECKPOINT_EVERY = 25  # at most this many focal works repeat after an interruption
CITATION_BATCH_SIZE = 1  # one focal work per request keeps Stage 3 responsive and isolates very highly cited papers
CITATION_CHECKPOINT_EVERY = 100  # persist every 100 focal works; at most this many repeat after an interruption

JOURNALS = [
    {'slug': 'mis_quarterly', 'name': 'MIS Quarterly', 'issn': '0276-7783'},
    {'slug': 'information_systems_research', 'name': 'Information Systems Research', 'issn': '1047-7047'},
    {'slug': 'journal_of_management_information_systems', 'name': 'Journal of Management Information Systems', 'issn': '0742-1222'},
    #{'slug': 'journal_of_the_association_for_information_systems', 'name': 'Journal of the Association for Information Systems', 'issn': '1536-9323'},
]

STATE_PATH = ROOT / 'atlas_state.json'
EXTERNAL_PATH = ROOT / 'atlas_external_papers.json'
EDGES_PATH = ROOT / 'atlas_graph_edges.json'

session = requests.Session()
session.headers.update({'User-Agent': 'InformationSystemsAtlas/1.0 (research dataset collection)'})

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def compact_id(value: str | None) -> str | None:
    if not value:
        return None
    return value.rstrip('/').rsplit('/', 1)[-1]

def json_load(path: Path, default: Any) -> Any:
    if not path.exists():
        return default
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def json_save(path: Path, payload: Any) -> None:
    """Atomic, Windows-safe JSON writing with retries for transient file locks."""
    last_error = None
    for attempt in range(MAX_WRITE_RETRIES):
        # A new temp name per attempt avoids reusing a stale .tmp file locked by a scanner or sync client.
        temporary = path.with_name(f'.{path.name}.{uuid4().hex}.tmp')
        try:
            with temporary.open('w', encoding='utf-8') as handle:
                # Compact JSON is materially faster and smaller for the large OpenAlex response files.
                json.dump(payload, handle, ensure_ascii=False, separators=(',', ':'))
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temporary, path)
            return
        except PermissionError as exc:
            last_error = exc
            time.sleep(min(4.0, 0.25 * (attempt + 1)))
        finally:
            if temporary.exists():
                try:
                    temporary.unlink()
                except PermissionError:
                    pass
    raise PermissionError(
        f'Could not replace {path.name} after {MAX_WRITE_RETRIES} attempts. '
        'Close any editor/preview/sync process holding that file, then re-run this cell; existing checkpoints remain valid.'
    ) from last_error

def initial_state() -> dict[str, Any]:
    return {
        'schema_version': '1.0',
        'started_at': utc_now(),
        'updated_at': utc_now(),
        'journal_source_ids': {},
        'journals': {journal['slug']: {
            'works_cursor': '*',
            'work_filter': 'locations.source.id',
            'works_complete': False,
            'references_processed_work_ids': [],
            'references_complete': False,
            'citations_processed_work_ids': [],
            'citations_complete': False,
        } for journal in JOURNALS},
        'graph_built_at': None,
        'active_quarterly_refresh': None,
        'last_successful_quarterly_refresh': None,
    }

state = json_load(STATE_PATH, initial_state())
state_migrated = False
for journal in JOURNALS:
    checkpoint = state.setdefault('journals', {}).setdefault(journal['slug'], initial_state()['journals'][journal['slug']])
    # Version 1.1 uses all locations, not only the primary location, for complete journal coverage.
    if checkpoint.get('work_filter') != 'locations.source.id':
        checkpoint['work_filter'] = 'locations.source.id'
        checkpoint['works_cursor'] = '*'
        checkpoint['works_complete'] = False
        checkpoint['references_complete'] = False
        checkpoint['citations_complete'] = False
        state_migrated = True
if state_migrated:
    state['updated_at'] = utc_now()
    json_save(STATE_PATH, state)

def save_state() -> None:
    state['updated_at'] = utc_now()
    json_save(STATE_PATH, state)

def api_get(endpoint: str, params: dict[str, Any] | None = None) -> dict[str, Any]:
    request_params = {'api_key': API_KEY}
    if params:
        request_params.update({key: value for key, value in params.items() if value is not None})
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(f'{API_URL}{endpoint}', params=request_params, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code == 429:
                delay = float(response.headers.get('Retry-After', min(60, 2 ** (attempt + 1))))
                time.sleep(delay)
                continue
            response.raise_for_status()
            time.sleep(REQUEST_PAUSE_SECONDS)
            return response.json()
        except requests.RequestException:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(min(60, 2 ** (attempt + 1)))
    raise RuntimeError('Unreachable retry guard')

def journal_path(journal: dict[str, str]) -> Path:
    return ROOT / f"{journal['slug']}_papers.json"

def empty_journal_data(journal: dict[str, str], source: dict[str, Any]) -> dict[str, Any]:
    return {
        'schema_version': '1.0',
        'atlas_scope': 'The Information Systems Atlas',
        'journal': {'name': journal['name'], 'issn': journal['issn'], 'openalex_source': source},
        'created_at': utc_now(),
        'updated_at': utc_now(),
        'works': {},
    }

def get_journal_data(journal: dict[str, str]) -> dict[str, Any]:
    source = state['journal_source_ids'][journal['slug']]
    return json_load(journal_path(journal), empty_journal_data(journal, source))

def save_journal_data(journal: dict[str, str], data: dict[str, Any]) -> None:
    data['updated_at'] = utc_now()
    json_save(journal_path(journal), data)

def decode_abstract(inverted_index: dict[str, list[int]] | None) -> str | None:
    if not inverted_index:
        return None
    positions = {position: token for token, places in inverted_index.items() for position in places}
    return ' '.join(positions[index] for index in sorted(positions)) or None

def source_summary(location: dict[str, Any] | None) -> dict[str, Any] | None:
    location = location or {}
    source = location.get('source') or {}
    if not source and not location:
        return None
    return {
        'id': compact_id(source.get('id')),
        'name': source.get('display_name'),
        'issn_l': source.get('issn_l'),
        'type': source.get('type'),
        'landing_page_url': location.get('landing_page_url'),
        'pdf_url': location.get('pdf_url'),
        'is_oa': location.get('is_oa'),
    }

def topic_summary(topic: dict[str, Any] | None) -> dict[str, Any] | None:
    if not topic:
        return None
    def level(name: str) -> dict[str, Any] | None:
        item = topic.get(name) or {}
        return {'id': compact_id(item.get('id')), 'name': item.get('display_name')} if item else None
    return {
        'id': compact_id(topic.get('id')),
        'name': topic.get('display_name'),
        'score': topic.get('score'),
        'subfield': level('subfield'),
        'field': level('field'),
        'domain': level('domain'),
    }

def authors_and_institutions(authorships: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    authors, institutions = [], {}
    for authorship in authorships or []:
        person = authorship.get('author') or {}
        affiliations = []
        for institution in authorship.get('institutions') or []:
            item = {
                'id': compact_id(institution.get('id')),
                'name': institution.get('display_name'),
                'ror': institution.get('ror'),
                'country_code': institution.get('country_code'),
                'type': institution.get('type'),
            }
            affiliations.append(item)
            institutions[item['id'] or item['name']] = item
        authors.append({
            'id': compact_id(person.get('id')),
            'name': person.get('display_name'),
            'orcid': person.get('orcid'),
            'position': authorship.get('author_position'),
            'is_corresponding': authorship.get('is_corresponding'),
            'institutions': affiliations,
        })
    return authors, list(institutions.values())

def normalized_work(work: dict[str, Any]) -> dict[str, Any]:
    authors, institutions = authors_and_institutions(work.get('authorships') or [])
    primary = source_summary(work.get('primary_location'))
    best_oa = source_summary(work.get('best_oa_location'))
    topics = [topic_summary(item) for item in (work.get('topics') or [])]
    return {
        'id': compact_id(work.get('id')),
        'openalex_url': work.get('id'),
        'doi': work.get('doi'),
        'title': work.get('title') or work.get('display_name'),
        'publication_date': work.get('publication_date'),
        'publication_year': work.get('publication_year'),
        'type': work.get('type'),
        'language': work.get('language'),
        'cited_by_count': work.get('cited_by_count'),
        'counts_by_year': work.get('counts_by_year') or [],
        'citation_normalized_percentile': work.get('citation_normalized_percentile'),
        'cited_by_percentile_year': work.get('cited_by_percentile_year'),
        'fwci': work.get('fwci'),
        'biblio': work.get('biblio') or {},
        'journal': primary,
        'best_open_access_location': best_oa,
        'open_access': work.get('open_access') or {},
        'link_to_paper': (primary or {}).get('landing_page_url') or work.get('doi') or (best_oa or {}).get('landing_page_url'),
        'abstract': decode_abstract(work.get('abstract_inverted_index')),
        'has_abstract': bool(work.get('abstract_inverted_index')),
        'authors': authors,
        'institutions': institutions,
        'primary_topic': topic_summary(work.get('primary_topic')),
        'topics': topics,
        'keywords': work.get('keywords') or [],
        'sustainable_development_goals': work.get('sustainable_development_goals') or [],
        'funders': work.get('funders') or [],
        'awards': work.get('awards') or [],
        'indexed_in': work.get('indexed_in') or [],
        'referenced_work_ids': [compact_id(item) for item in (work.get('referenced_works') or []) if compact_id(item)],
        'citing_work_ids': [],
        'openalex_updated_date': work.get('updated_date'),
        'openalex_raw': work,
    }

def external_summary(work: dict[str, Any]) -> dict[str, Any]:
    primary = source_summary(work.get('primary_location'))
    return {
        'id': compact_id(work.get('id')),
        'title': work.get('title') or work.get('display_name'),
        'journal': (primary or {}).get('name'),
        'journal_source_id': (primary or {}).get('id'),
        'publication_year': work.get('publication_year'),
        'doi': work.get('doi'),
        'link_to_paper': (primary or {}).get('landing_page_url') or work.get('doi'),
    }

def chunks(values: Iterable[str], size: int) -> Iterable[list[str]]:
    values = list(values)
    for index in range(0, len(values), size):
        yield values[index:index + size]

def resolve_sources() -> None:
    for journal in JOURNALS:
        if journal['slug'] in state.get('journal_source_ids', {}):
            continue
        source = api_get(f"/sources/issn:{journal['issn']}")
        state['journal_source_ids'][journal['slug']] = {
            'id': compact_id(source.get('id')),
            'openalex_url': source.get('id'),
            'display_name': source.get('display_name'),
            'issn_l': source.get('issn_l'),
            'issn': source.get('issn'),
            'homepage_url': source.get('homepage_url'),
            'works_count': source.get('works_count'),
        }
        save_state()

def focal_source_ids() -> set[str]:
    return {source['id'] for source in state['journal_source_ids'].values()}

def work_source_ids(work: dict[str, Any]) -> set[str]:
    locations = list(work.get('locations') or [])
    if work.get('primary_location'):
        locations.append(work['primary_location'])
    return {item['id'] for location in locations if (item := source_summary(location)) and item.get('id')}

def is_focal_work(work: dict[str, Any]) -> bool:
    return bool(work_source_ids(work).intersection(focal_source_ids()))

def fetch_works_by_ids(work_ids: Iterable[str]) -> list[dict[str, Any]]:
    found = []
    for batch in chunks(sorted(set(work_ids)), 50):
        if not batch:
            continue
        response = api_get('/works', {'filter': 'ids.openalex:' + '|'.join(batch), 'per_page': len(batch)})
        found.extend(response.get('results') or [])
    return found

def load_external() -> dict[str, Any]:
    return json_load(EXTERNAL_PATH, {
        'schema_version': '1.0',
        'purpose': 'Minimal records for cited/citing works outside the four focal IS journals',
        'updated_at': utc_now(),
        'works': {},
    })

def add_external_records(records: Iterable[dict[str, Any]], external: dict[str, Any] | None = None) -> bool:
    should_save = external is None
    external = external if external is not None else load_external()
    changed = False
    for record in records:
        work_id = compact_id(record.get('id'))
        if work_id and not is_focal_work(record):
            external['works'][work_id] = external_summary(record)
            changed = True
    if changed and should_save:
        external['updated_at'] = utc_now()
        json_save(EXTERNAL_PATH, external)
    return changed

print(f'Notebook root: {ROOT}')
print('OpenAlex key located. Ready to collect.')


c:\Users\moham\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook root: C:\Drive\DESB\Atlas
OpenAlex key located. Ready to collect.


In [2]:
# Collection and graph build — run this cell. Re-run it to resume.
# Set this to True only when starting a planned three-month refresh; turn it back to False afterwards.
START_QUARTERLY_REFRESH = False

def start_quarterly_refresh() -> None:
    """Prepare a complete, resumable refresh without deleting the previous successful records first."""
    if state.get('active_quarterly_refresh'):
        print('A quarterly refresh is already in progress; resuming it instead of resetting it.')
        return
    missing_sources = [journal['name'] for journal in JOURNALS if journal['slug'] not in state.get('journal_source_ids', {})]
    if missing_sources:
        raise RuntimeError('Run a normal collection first so source IDs are resolved: ' + ', '.join(missing_sources))
    refresh_started_at = utc_now()
    for journal in JOURNALS:
        checkpoint = state['journals'][journal['slug']]
        checkpoint.update({
            'works_cursor': '*',
            'works_complete': False,
            'references_processed_work_ids': [],
            'references_complete': False,
            'citations_processed_work_ids': [],
            'citations_complete': False,
            'refresh_seen_work_ids': [],
            'refresh_started_at': refresh_started_at,
        })
        # Incoming citations are rebuilt from OpenAlex, so removed citation links cannot survive the refresh.
        data = get_journal_data(journal)
        for record in data['works'].values():
            record['citing_work_ids'] = []
        save_journal_data(journal, data)
    state['active_quarterly_refresh'] = refresh_started_at
    save_state()
    print('Quarterly refresh prepared. Keep START_QUARTERLY_REFRESH=False on subsequent reruns to resume safely.')

def finish_quarterly_refresh() -> None:
    if not state.get('active_quarterly_refresh'):
        return
    if not all(state['journals'][journal['slug']]['citations_complete'] for journal in JOURNALS):
        return
    prune_external_papers_to_current_graph()
    state['last_successful_quarterly_refresh'] = utc_now()
    state['active_quarterly_refresh'] = None
    for journal in JOURNALS:
        state['journals'][journal['slug']].pop('refresh_seen_work_ids', None)
        state['journals'][journal['slug']].pop('refresh_started_at', None)
    save_state()
    print('Quarterly refresh completed successfully.')

def collect_journal_works(journal: dict[str, str]) -> None:
    checkpoint = state['journals'][journal['slug']]
    if checkpoint['works_complete']:
        print(f"✓ {journal['name']}: focal works already complete")
        return
    source_id = state['journal_source_ids'][journal['slug']]['id']
    data = get_journal_data(journal)
    cursor = checkpoint.get('works_cursor') or '*'
    refresh_seen = set(checkpoint.get('refresh_seen_work_ids') or [])
    bar = tqdm(desc=f"{journal['name']} — focal works", unit='works')
    try:
        while cursor:
            page = api_get('/works', {
                'filter': f'locations.source.id:{source_id}',
                'per_page': PER_PAGE,
                'cursor': cursor,
                'sort': 'publication_date:asc',
            })
            if bar.total is None:
                bar.total = (page.get('meta') or {}).get('count')
            results = page.get('results') or []
            for work in results:
                record = normalized_work(work)
                work_id = record['id']
                previous = data['works'].get(work_id, {})
                record['citing_work_ids'] = previous.get('citing_work_ids', [])
                data['works'][work_id] = record
                if state.get('active_quarterly_refresh'):
                    refresh_seen.add(work_id)
            bar.update(len(results))
            cursor = (page.get('meta') or {}).get('next_cursor')
            checkpoint['works_cursor'] = cursor
            if state.get('active_quarterly_refresh'):
                checkpoint['refresh_seen_work_ids'] = sorted(refresh_seen)
            save_journal_data(journal, data)
            save_state()
            if not results or not cursor:
                break
        if state.get('active_quarterly_refresh'):
            # Remove a focal record only after the full fresh listing succeeds.
            for stale_work_id in set(data['works']).difference(refresh_seen):
                del data['works'][stale_work_id]
            checkpoint['refresh_seen_work_ids'] = sorted(refresh_seen)
            save_journal_data(journal, data)
        checkpoint['works_complete'] = True
        checkpoint['works_cursor'] = None
        save_state()
    finally:
        bar.close()

def collect_external_references(journal: dict[str, str]) -> None:
    checkpoint = state['journals'][journal['slug']]
    if checkpoint['references_complete']:
        print(f"✓ {journal['name']}: referenced external papers already complete")
        return
    data = get_journal_data(journal)
    done = set(checkpoint.get('references_processed_work_ids') or [])
    pending = [work_id for work_id in data['works'] if work_id not in done]
    external = load_external()
    external_dirty = False
    bar = tqdm(total=len(pending), desc=f"{journal['name']} — cited sources", unit='focal works')
    try:
        for processed_count, work_id in enumerate(pending, start=1):
            reference_ids = data['works'][work_id].get('referenced_work_ids') or []
            # A batch lookup yields title/journal only for external neighbors; focal works stay in their journal files.
            external_dirty = add_external_records(fetch_works_by_ids(reference_ids), external) or external_dirty
            done.add(work_id)
            if processed_count % REFERENCE_CHECKPOINT_EVERY == 0:
                if external_dirty:
                    external['updated_at'] = utc_now()
                    json_save(EXTERNAL_PATH, external)
                    external_dirty = False
                checkpoint['references_processed_work_ids'] = sorted(done)
                save_state()
            bar.update(1)
        if external_dirty:
            external['updated_at'] = utc_now()
            json_save(EXTERNAL_PATH, external)
        checkpoint['references_processed_work_ids'] = sorted(done)
        checkpoint['references_complete'] = True
        save_state()
    finally:
        bar.close()

def collect_citing_papers(journal: dict[str, str]) -> None:
    checkpoint = state['journals'][journal['slug']]
    if checkpoint['citations_complete']:
        print(f"✓ {journal['name']}: citing papers already complete")
        return
    data = get_journal_data(journal)
    done = set(checkpoint.get('citations_processed_work_ids') or [])
    pending = [work_id for work_id in data['works'] if work_id not in done]
    external = load_external()
    batches_since_checkpoint = 0
    journal_dirty = False
    external_dirty = False
    bar = tqdm(total=len(pending), desc=f"{journal['name']} — citing papers", unit='focal works')
    try:
        # Save only at durable batch checkpoints. Rewriting a whole journal file after each API page is prohibitively slow.
        for target_ids in chunks(pending, CITATION_BATCH_SIZE):
            target_id = target_ids[0]
            cursor = '*'
            page_count = 0
            received_count = 0
            while cursor:
                page = api_get('/works', {
                    'filter': 'cites:' + target_id,
                    'per_page': PER_PAGE,
                    'cursor': cursor,
                    # External neighbors only need linkable display metadata; requesting full work objects is much slower.
                    'select': 'id,title,doi,publication_year,primary_location,locations',
                })
                results = page.get('results') or []
                external_dirty = add_external_records(results, external) or external_dirty
                incoming = set(data['works'][target_id].get('citing_work_ids') or [])
                incoming.update(compact_id(work.get('id')) for work in results if compact_id(work.get('id')))
                data['works'][target_id]['citing_work_ids'] = sorted(incoming)
                page_count += 1
                received_count += len(results)
                target_title = data['works'][target_id].get('title') or 'Untitled work'
                bar.set_postfix_str(f"{target_title[:34]} | {received_count:,} citing records")
                cursor = (page.get('meta') or {}).get('next_cursor')
                if not results or not cursor:
                    break
            done.update(target_ids)
            journal_dirty = True
            batches_since_checkpoint += 1
            bar.update(len(target_ids))
            if batches_since_checkpoint >= CITATION_CHECKPOINT_EVERY:
                if external_dirty:
                    external['updated_at'] = utc_now()
                    json_save(EXTERNAL_PATH, external)
                    external_dirty = False
                if journal_dirty:
                    save_journal_data(journal, data)
                    journal_dirty = False
                checkpoint['citations_processed_work_ids'] = sorted(done)
                save_state()
                batches_since_checkpoint = 0
        if external_dirty:
            external['updated_at'] = utc_now()
            json_save(EXTERNAL_PATH, external)
        if journal_dirty:
            save_journal_data(journal, data)
        checkpoint['citations_processed_work_ids'] = sorted(done)
        checkpoint['citations_complete'] = True
        save_state()
    finally:
        bar.close()

def doi_resolver_url(doi: str | None) -> str | None:
    if not doi:
        return None
    return doi if doi.startswith('http') else f"https://doi.org/{doi.removeprefix('doi:')}"

def build_graph_edges() -> None:
    edges = set()
    focal_count = 0
    # The shared node lookup lets every edge carry DOI/link metadata for both ends.
    node_lookup = dict(load_external().get('works') or {})
    for journal in JOURNALS:
        data = get_journal_data(journal)
        for target_id, record in data['works'].items():
            node_lookup[target_id] = record  # focal record is richer than a minimal external record
            focal_count += 1
            for reference_id in record.get('referenced_work_ids') or []:
                edges.add((target_id, reference_id))
            for citing_id in record.get('citing_work_ids') or []:
                edges.add((citing_id, target_id))
    graph_edges = []
    for source_id, target_id in sorted(edges):
        source = node_lookup.get(source_id, {})
        target = node_lookup.get(target_id, {})
        graph_edges.append({
            'source_id': source_id,
            'source_doi': source.get('doi'),
            'source_doi_url': doi_resolver_url(source.get('doi')),
            'target_id': target_id,
            'target_doi': target.get('doi'),
            'target_doi_url': doi_resolver_url(target.get('doi')),
            'relationship': 'cites',
        })
    payload = {
        'schema_version': '1.0',
        'description': 'Directed citation edges: source_id cites target_id. Node metadata lives in the journal and external JSON files.',
        'built_at': utc_now(),
        'focal_work_count': focal_count,
        'edge_count': len(edges),
        'edges': graph_edges,
    }
    json_save(EDGES_PATH, payload)
    state['graph_built_at'] = payload['built_at']
    save_state()
    print(f"Graph ready: {focal_count:,} focal papers and {len(edges):,} citation edges.")

def prune_external_papers_to_current_graph() -> None:
    """After a full refresh, remove external records that no longer occur in the current graph."""
    graph = json_load(EDGES_PATH, {})
    linked_ids = {edge['source_id'] for edge in graph.get('edges', [])}
    linked_ids.update(edge['target_id'] for edge in graph.get('edges', []))
    focal_ids = set()
    for journal in JOURNALS:
        focal_ids.update(get_journal_data(journal)['works'])
    external = load_external()
    pruned = {work_id: record for work_id, record in external['works'].items() if work_id in linked_ids and work_id not in focal_ids}
    if len(pruned) != len(external['works']):
        external['works'] = pruned
        external['updated_at'] = utc_now()
        json_save(EXTERNAL_PATH, external)

resolve_sources()
if START_QUARTERLY_REFRESH:
    start_quarterly_refresh()
elif state.get('active_quarterly_refresh'):
    print('Resuming the quarterly refresh already recorded in atlas_state.json.')

try:
    print('\n=== Stage 1 of 3: focal papers in all four journals ===')
    for journal in JOURNALS:
        collect_journal_works(journal)

    print('\n=== Stage 2 of 3: external papers referenced by all focal papers ===')
    for journal in JOURNALS:
        collect_external_references(journal)

    print('\n=== Stage 3 of 3: papers citing all focal papers ===')
    for journal in JOURNALS:
        collect_citing_papers(journal)
    build_graph_edges()
    finish_quarterly_refresh()
except KeyboardInterrupt:
    save_state()
    print('Stopped safely. Re-run this cell to continue from atlas_state.json.')
    raise

print('All collection stages are complete. You can now use the JSON files to build the Atlas dashboard.')


Quarterly refresh prepared. Keep START_QUARTERLY_REFRESH=False on subsequent reruns to resume safely.

=== Stage 1 of 3: focal papers in all four journals ===


Information Systems Research — focal works: 100%|██████████| 2031/2031 [01:22<00:00, 24.70works/s]
Journal of Management Information Systems — focal works: 100%|██████████| 1881/1881 [01:35<00:00, 19.63works/s]



=== Stage 2 of 3: external papers referenced by all focal papers ===


Information Systems Research — cited sources: 100%|██████████| 2031/2031 [51:00<00:00,  1.51s/focal works]  
Journal of Management Information Systems — cited sources: 100%|██████████| 1881/1881 [47:11<00:00,  1.51s/focal works]  



=== Stage 3 of 3: papers citing all focal papers ===


Information Systems Research — citing papers: 100%|██████████| 2031/2031 [38:26<00:00,  1.14s/focal works, Helping Oneself by Helping a Rival | 0 citing records]     
Journal of Management Information Systems — citing papers: 100%|██████████| 1881/1881 [30:53<00:00,  1.02focal works/s, Correction | 0 citing records]                                                   


Graph ready: 6,753 focal papers and 1,529,177 citation edges.
Quarterly refresh completed successfully.
All collection stages are complete. You can now use the JSON files to build the Atlas dashboard.


In [4]:
# Optional: compact status report (safe to run at any time)
state = json_load(STATE_PATH, initial_state())
for journal in JOURNALS:
    checkpoint = state['journals'][journal['slug']]
    data = get_journal_data(journal) if journal['slug'] in state.get('journal_source_ids', {}) else {'works': {}}
    print(f"{journal['name']}: {len(data['works']):,} focal works | works={checkpoint['works_complete']} | references={checkpoint['references_complete']} | citing={checkpoint['citations_complete']}")
external = load_external()
print(f"External neighbor records: {len(external['works']):,}")

# Coverage QA: compare the saved incoming IDs to OpenAlex's snapshot citation count.
# Small gaps can occur while OpenAlex updates its derived cited_by_count; large gaps merit a rerun.
for journal in JOURNALS:
    checkpoint = state['journals'][journal['slug']]
    if not checkpoint['citations_complete'] or journal['slug'] not in state.get('journal_source_ids', {}):
        continue
    data = get_journal_data(journal)
    expected = sum((record.get('cited_by_count') or 0) for record in data['works'].values())
    collected = sum(len(record.get('citing_work_ids') or []) for record in data['works'].values())
    print(f"{journal['name']} incoming-citation QA: {collected:,} saved IDs vs {expected:,} OpenAlex snapshot count")
if EDGES_PATH.exists():
    print(f"Citation edges: {json_load(EDGES_PATH, {})['edge_count']:,}")


MIS Quarterly: 2,841 focal works | works=True | references=True | citing=True
Information Systems Research: 2,031 focal works | works=True | references=True | citing=True
Journal of Management Information Systems: 1,881 focal works | works=True | references=True | citing=True
External neighbor records: 538,790
MIS Quarterly incoming-citation QA: 729,320 saved IDs vs 730,511 OpenAlex snapshot count
Information Systems Research incoming-citation QA: 294,949 saved IDs vs 295,440 OpenAlex snapshot count
Journal of Management Information Systems incoming-citation QA: 232,735 saved IDs vs 232,989 OpenAlex snapshot count
Citation edges: 1,529,177
